# Fake Jobs – Experiment 1: AnoLLM
- LoRA-Finetuning Qwen2.5-0.5B auf serialisierten Zeilen, Score = NLL
- Unsupervised: Training auf dem **Train-Split ohne Labels**; lesbare Rohwerte + Freitexte

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import mlflow
import torch.distributed as dist
from torch.utils.data import DataLoader, SequentialSampler
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc

sys.path.insert(0, "../../anollm_src")
import anollm.anollm_trainer
from anollm.anollm import AnoLLM

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Rohdaten aufbereiten
- ID/Label droppen, NaN behandeln (Kategorien → "missing", Texte → ""); übrige Spalten bleiben roh und lesbar

In [ ]:
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
cat_cols = ["location", "department", "salary_range", "employment_type",
            "required_experience", "required_education", "industry", "function"]

raw = pd.read_csv("../../data/raw/fake_job_postings.csv")
raw["row_id"] = raw["job_id"].astype("int64")
lab = pd.Series(raw["fraudulent"].values, index=raw["row_id"].values)  # Outlier = 1
raw = raw.drop(columns=["job_id", "fraudulent"])
raw[cat_cols] = raw[cat_cols].fillna("missing")
raw[text_cols] = raw[text_cols].fillna("")
feat = raw.set_index("row_id")

## Split laden
- Dieselbe Split-Datei wie alle übrigen Modelle; es wird nicht neu gesplittet

In [ ]:
split = pd.read_csv(f"../../data/splits/split_fake_jobs_seed{SEED}.csv")
train_ids = split.loc[split["split"] == "train", "row_id"].values
test_ids = split.loc[split["split"] == "test", "row_id"].values

df_train = feat.loc[train_ids].reset_index(drop=True)
df_test = feat.loc[test_ids].reset_index(drop=True)
y_train, y_test = lab.loc[train_ids].values, lab.loc[test_ids].values
k = round(len(y_test) * y_train.mean())
print("train", df_train.shape, "test", df_test.shape, "| Outlier-Rate Test:", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_1")

## Single-GPU-Setup (kein DDP/NCCL)
- Verteilte Env-Variablen entfernen und den Trainer-Dataloader patchen → kein DistributedDataParallel

In [ ]:
for key in ["LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(key, None)
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()

def single_gpu_get_train_dataloader(self):
    return DataLoader(self.train_dataset, batch_size=self._train_batch_size,
                      sampler=SequentialSampler(self.train_dataset),
                      collate_fn=self.data_collator, drop_last=True)

anollm.anollm_trainer.AnoLLMTrainer.get_train_dataloader = single_gpu_get_train_dataloader
print("Single-GPU-Modus aktiv, Trainer gepatcht.")

## AnoLLM trainieren & Scores (NLL)
- `max_length_dict` begrenzt die Textspalten (Token-Budget); Score = NLL in nativer Orientierung (höher = anomaler)
- Schwelle für den Classification Report: Top-k Scores mit k = Testgröße × Outlier-Rate im Train

In [ ]:
model = AnoLLM(llm="Qwen/Qwen2.5-0.5B", efficient_finetuning="lora", textual_columns=text_cols,
               max_length_dict={c: 64 for c in text_cols}, batch_size=2, max_steps=1000, learning_rate=5e-4)

t0 = time.perf_counter()
model.fit(df_train)
scores = model.decision_function(df_test, n_permutations=8, batch_size=2, device="cuda").mean(axis=1)
runtime = time.perf_counter() - t0

scores = np.asarray(scores).astype(float)
prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)
auroc = roc_auc_score(y_test, scores)
pred = (scores >= np.sort(scores)[-k]).astype(int)

with mlflow.start_run(run_name="anollm"):
    mlflow.log_params({"llm": "Qwen/Qwen2.5-0.5B", "max_steps": 1000, "n_permutations": 8, "seed": SEED})
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"anollm: AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} time={runtime:.1f}s")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))